# B2-020-language-transformers — Practice p08 — Solution

**Type:** constrained-coding · **Difficulty:** core · **Concepts:** language-transformer, causal-language-modeling

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260812`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C11-neural-training`, `B2-019-attention-transformers`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb).

## Solution

Shift before the model call and preserve both batch and sequence axes.

In [ ]:
import torch
from torch import nn

ATOL = RTOL = 1e-6

def build_causal_batch(tokens, model):
    inputs = tokens[:, :-1]
    targets = tokens[:, 1:]
    logits = model(inputs, mask_mode="causal")
    return inputs, targets, logits

class ProbeModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(12, 8)
        self.head = nn.Linear(8, 12)
        self.modes = []
    def forward(self, ids, *, mask_mode):
        self.modes.append(mask_mode)
        return self.head(self.embedding(ids))

torch.manual_seed(20260812)
tokens = torch.tensor([[2,4,6,3,0,0,0,0], [2,5,7,8,11,3,0,0]], dtype=torch.int64)
model = ProbeModel()
inputs, targets, logits = build_causal_batch(tokens, model)

### Answer check

In [ ]:
assert inputs.shape == targets.shape == (2, 7)
assert logits.shape == (2, 7, 12) and logits.dtype == torch.float32
assert model.modes == ["causal"]
assert targets[0, 0].item() == 4 and targets[1, -1].item() == 0
assert torch.equal(inputs, tokens[:, :-1]) and torch.equal(targets, tokens[:, 1:])